# Лабораторная работа №3. Проведение исследований с решающим деревом

**Студент:** Чибугаев Илья Алексеевич  
**Вариант:** 22  
**Случайное зерно:** 22

## Исходные данные

- **Классификация:** [Ionosphere, UCI](https://archive.ics.uci.edu/dataset/52/ionosphere) — 351 объект, 34 числовых признака; `bad=1` обозначает сигнал без структуры в ионосфере.
- **Регрессия:** [Concrete Compressive Strength, UCI](https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength) — 1030 составов бетона, 8 признаков, целевая переменная — прочность в МПа.

Датасеты выбраны для разных практических задач: контроль качества радиосигнала и предварительная оценка рецептуры бетона. Оба набора небольшие, содержат только числовые признаки и позволяют сопоставить алгоритмы без изменения задачи между лабораторными.

CART рекурсивно делит пространство признаков, минимизируя неоднородность листьев: индекс Джини для классификации и дисперсию ответа для регрессии. Неограниченное дерево склонно запоминать обучающую выборку.

## Протокол и метрики

Для обеих задач используется фиксированное разбиение **80% / 20%** с `random_state=22`; в классификации разбиение стратифицировано. Подбор параметров выполняется только внутри обучающей выборки. Все четыре конфигурации (sklearn / собственная, базовая / улучшенная) оцениваются на одном и том же тестовом наборе.

- **Классификация:** основная метрика — **ROC-AUC**, поскольку она оценивает ранжирование и не зависит от одного порога; **F1** контролирует качество по классу `bad`, а **accuracy** показывает общую долю верных ответов.
- **Регрессия:** основная метрика — **RMSE (МПа)**, штрафующая крупные ошибки; **MAE (МПа)** даёт типичную абсолютную ошибку, **R²** — долю объяснённой дисперсии.


## 1. Подготовка

Код импортирует только общие функции и фиксирует настройки отображения. Каждая следующая ячейка имеет отдельный комментарий.

In [1]:
# Настраиваем воспроизводимое окружение и импортируем общий код эксперимента.
from pathlib import Path
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.experiments import TaskExperiment, LAB_NAMES

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"Корень проекта: {ROOT}")

Корень проекта: /Users/finleyl/Documents/kimi/tasks/2026-11-10/20-02-04-6933ea61/ml-labs-1-5


### Загрузка и проверка разбиений

Проверяем размеры, отсутствие пропусков, баланс классов и диапазон целевой переменной. Разбиения одинаковы во всех пяти работах.

In [2]:
# Создаём эксперименты для двух задач и выводим контрольные характеристики данных.
experiments = {task: TaskExperiment(3, task) for task in ("classification", "regression")}
for task, experiment in experiments.items():
    data = experiment.data
    print(f"{task}: {data.name}")
    print(data.practical_task)
    print(f"train={data.X_train.shape}, test={data.X_test.shape}, пропуски={int(data.X.isna().sum().sum())}")
    if task == "classification":
        print(f"Доля bad: train={data.y_train.mean():.3f}, test={data.y_test.mean():.3f}")
    else:
        print(data.y.describe().to_string())
    print()

classification: Ionosphere (UCI)
По сигналам фазированной антенной решётки отличать радиосигналы со структурой в ионосфере от проходящих сигналов.
train=(280, 34), test=(71, 34), пропуски=0
Доля bad: train=0.361, test=0.352

regression: Concrete Compressive Strength (UCI)
По составу смеси и возрасту бетона прогнозировать прочность на сжатие в МПа для предварительной оценки рецептуры.
train=(824, 8), test=(206, 8), пропуски=0
count    1030.0000
mean       35.8178
std        16.7057
min         2.3318
25%        23.7071
50%        34.4428
75%        46.1363
max        82.5992



## 2. Базовый бейзлайн sklearn

Бейзлайн использует параметры алгоритма по умолчанию и только медианное заполнение пропусков. Это точка отсчёта, а не ожидаемый победитель.

In [3]:
# Обучаем базовые sklearn-модели и оцениваем их на отложенной выборке.
for experiment in experiments.values():
    experiment.fit_sklearn_baseline()
baseline_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(baseline_results.drop(columns=["params"]))

classification | sklearn | baseline: accuracy=0.9014, f1=0.8571, roc_auc=0.8874; 0.00 s
regression | sklearn | baseline: rmse=5.7453, mae=3.8035, r2=0.8895; 0.00 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,3,Решающее дерево,classification,sklearn,baseline,0.9014,0.8571,0.8874,NaN,NaN,NaN,0.0034
1,3,Решающее дерево,regression,sklearn,baseline,NaN,NaN,NaN,5.7453,3.8035,0.8895,0.0029


## 3. Улучшение бейзлайна

### Проверяемые гипотезы

1. Ограничение глубины и минимального числа объектов в листе должно уменьшить переобучение.
2. Масштабирование не требуется: пороговые разбиения инвариантны к монотонному преобразованию каждого признака.
3. Оптимальные max_depth и min_samples_leaf выбираются только по обучающей части на 5-блочной CV.

Метрика подбора — ROC-AUC для классификации и RMSE для регрессии. Лучшие параметры затем один раз оцениваются на тесте.

In [4]:
# Запускаем GridSearchCV и показываем пять лучших конфигураций для каждой задачи.
for experiment in experiments.values():
    experiment.fit_sklearn_improved()
    print(f"{experiment.task}: лучшие параметры = {experiment.search.best_params_}")
    display(experiment.cv_table().head())

classification | sklearn | improved: accuracy=0.8732, f1=0.8235, roc_auc=0.9335; 1.30 s
classification: лучшие параметры = {'model__max_depth': 5, 'model__min_samples_leaf': 7}


,params,cv_roc_auc,std_roc_auc,mean_fit_time
0,"{'model__max_depth': 5, 'model__min_samples_le...",0.9126,0.0410,0.0046
1,"{'model__max_depth': 8, 'model__min_samples_le...",0.9036,0.0488,0.0041
2,"{'model__max_depth': None, 'model__min_samples...",0.9036,0.0488,0.0035
3,"{'model__max_depth': None, 'model__min_samples...",0.8937,0.0465,0.0039
4,"{'model__max_depth': 8, 'model__min_samples_le...",0.8932,0.0535,0.0043


regression | sklearn | improved: rmse=5.7453, mae=3.8035, r2=0.8895; 0.04 s
regression: лучшие параметры = {'model__max_depth': None, 'model__min_samples_leaf': 1}


,params,cv_rmse,std_rmse,mean_fit_time
0,"{'model__max_depth': None, 'model__min_samples...",6.7952,0.6779,0.0033
1,"{'model__max_depth': None, 'model__min_samples...",7.0093,0.5352,0.0029
2,"{'model__max_depth': 8, 'model__min_samples_le...",7.1351,0.3608,0.0029
3,"{'model__max_depth': 8, 'model__min_samples_le...",7.3234,0.3899,0.0026
4,"{'model__max_depth': None, 'model__min_samples...",7.7926,0.6389,0.0027


### Сравнение базового и улучшенного sklearn-бейзлайна

Положительная разница ROC-AUC/F1/accuracy/R² означает улучшение; для RMSE/MAE улучшение соответствует отрицательной разнице.

In [5]:
# Формируем таблицу изменений метрик между улучшенной и базовой sklearn-моделями.
metric_names = {"classification": ["roc_auc", "f1", "accuracy"], "regression": ["rmse", "mae", "r2"]}
for task, experiment in experiments.items():
    rows = experiment.results.set_index("stage")
    metrics = metric_names[task]
    comparison = rows.loc[["baseline", "improved"], metrics].T
    comparison["change"] = comparison["improved"] - comparison["baseline"]
    print(task)
    display(comparison)

classification


stage,baseline,improved,change
roc_auc,0.8874,0.9335,0.0461
f1,0.8571,0.8235,-0.0336
accuracy,0.9014,0.8732,-0.0282


regression


stage,baseline,improved,change
rmse,5.7453,5.7453,0.0
mae,3.8035,3.8035,0.0
r2,0.8895,0.8895,0.0


## 4. Собственная реализация

На каждом узле перебираются допустимые пороги отсортированных признаков. Кумулятивные суммы дают вычисление Джини и SSE без повторного прохода по каждому кандидату.

Исходный код находится в `src/models/`. Сначала используются базовые параметры, затем техники и параметры улучшенного бейзлайна.

In [6]:
# Обучаем собственные базовые модели на тех же преобразованных данных, что и sklearn-бейзлайн.
for experiment in experiments.values():
    experiment.fit_custom_baseline()
custom_baseline_results = pd.concat([
    experiment.results[experiment.results["implementation"] == "from_scratch"]
    for experiment in experiments.values()
], ignore_index=True)
display(custom_baseline_results.drop(columns=["params"]))

classification | from_scratch | baseline: accuracy=0.8873, f1=0.8400, roc_auc=0.8765; 0.01 s
regression | from_scratch | baseline: rmse=5.4395, mae=3.5648, r2=0.9009; 0.07 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,3,Решающее дерево,classification,from_scratch,baseline,0.8873,0.84,0.8765,NaN,NaN,NaN,0.0097
1,3,Решающее дерево,regression,from_scratch,baseline,NaN,NaN,NaN,5.4395,3.5648,0.9009,0.0735


### Собственная модель с улучшенным бейзлайном

Преобразования обучены только на train. Для собственной модели переносятся выбранные на CV параметры; исключение по числу деревьев случайного леса явно указано в гипотезах.

In [7]:
# Добавляем улучшенный препроцессинг и выбранные параметры в собственные реализации.
for experiment in experiments.values():
    experiment.fit_custom_improved()
all_lab_results = pd.concat([experiment.results for experiment in experiments.values()], ignore_index=True)
display(all_lab_results.drop(columns=["params"]))

classification | from_scratch | improved: accuracy=0.8592, f1=0.7917, roc_auc=0.9270; 0.00 s


regression | from_scratch | improved: rmse=5.4395, mae=3.5648, r2=0.9009; 0.07 s


,lab,algorithm,task,implementation,stage,accuracy,f1,roc_auc,rmse,mae,r2,fit_seconds
0,3,Решающее дерево,classification,sklearn,baseline,0.9014,0.8571,0.8874,NaN,NaN,NaN,0.0034
1,3,Решающее дерево,classification,sklearn,improved,0.8732,0.8235,0.9335,NaN,NaN,NaN,1.3034
2,3,Решающее дерево,classification,from_scratch,baseline,0.8873,0.8400,0.8765,NaN,NaN,NaN,0.0097
3,3,Решающее дерево,classification,from_scratch,improved,0.8592,0.7917,0.9270,NaN,NaN,NaN,0.0047
4,3,Решающее дерево,regression,sklearn,baseline,NaN,NaN,NaN,5.7453,3.8035,0.8895,0.0029
5,3,Решающее дерево,regression,sklearn,improved,NaN,NaN,NaN,5.7453,3.8035,0.8895,0.0447
6,3,Решающее дерево,regression,from_scratch,baseline,NaN,NaN,NaN,5.4395,3.5648,0.9009,0.0735
7,3,Решающее дерево,regression,from_scratch,improved,NaN,NaN,NaN,5.4395,3.5648,0.9009,0.0727


## 6. Сохранение результатов и выводы

Вывод формируется по фактическим значениям, а не по предположению, что улучшение обязано помочь. На маленькой тестовой выборке небольшие различия не следует трактовать как статистически доказанное превосходство.

In [8]:
# Сохраняем четыре результата на задачу и печатаем аргументированное заключение по каждой задаче.
output_path = ROOT / "results" / "lab03_results.csv"
output_path.parent.mkdir(exist_ok=True)
all_lab_results.to_csv(output_path, index=False)

for task, experiment in experiments.items():
    frame = experiment.results.set_index(["implementation", "stage"])
    metrics = metric_names[task]
    sklearn_base = frame.loc[("sklearn", "baseline"), metrics]
    sklearn_best = frame.loc[("sklearn", "improved"), metrics]
    custom_base = frame.loc[("from_scratch", "baseline"), metrics]
    custom_best = frame.loc[("from_scratch", "improved"), metrics]
    primary = "roc_auc" if task == "classification" else "rmse"
    higher_is_better = primary != "rmse"
    sklearn_change = sklearn_best[primary] - sklearn_base[primary]
    custom_change = custom_best[primary] - custom_base[primary]
    sklearn_verdict = "улучшился" if (sklearn_change > 0) == higher_is_better else "не улучшился"
    custom_verdict = "улучшился" if (custom_change > 0) == higher_is_better else "не улучшился"
    print(f"{task}: sklearn {primary} {sklearn_base[primary]:.4f} → {sklearn_best[primary]:.4f} ({sklearn_verdict}); "
          f"своя реализация {custom_base[primary]:.4f} → {custom_best[primary]:.4f} ({custom_verdict}).")
    print(f"Лучшие параметры sklearn: {experiment.search.best_params_}")
print(f"Результаты сохранены: {output_path.relative_to(ROOT)}")

classification: sklearn roc_auc 0.8874 → 0.9335 (улучшился); своя реализация 0.8765 → 0.9270 (улучшился).
Лучшие параметры sklearn: {'model__max_depth': 5, 'model__min_samples_leaf': 7}
regression: sklearn rmse 5.7453 → 5.7453 (улучшился); своя реализация 5.4395 → 5.4395 (улучшился).
Лучшие параметры sklearn: {'model__max_depth': None, 'model__min_samples_leaf': 1}
Результаты сохранены: results/lab03_results.csv
